In [42]:
import sqlite3 as sql

conn = sql.connect("data/bookStore.db")
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

# Categories table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Categories (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT UNIQUE NOT NULL
)
""")

# Books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Books (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Title TEXT NOT NULL,
    CategoryID INTEGER NOT NULL,
    Price_GBP REAL,
    Price_INR REAL,
    Star_rating INTEGER,
    in_stock INTEGER,
    FOREIGN KEY (CategoryID) REFERENCES Categories(ID)
)
""")

conn.commit()

print("Database and Tables(Books and Category) Created successfully")

Database and Tables(Books and Category) Created successfully


In [43]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
cursor.fetchall()

[('Categories',), ('sqlite_sequence',), ('Books',)]

In [44]:
import pandas as pd
books=pd.read_csv("data/clean_books.csv")
books.info()
Categories=books['Category'].unique()
Categories

<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        66 non-null     str    
 1   Price_GBP    66 non-null     float64
 2   Price_INR    66 non-null     float64
 3   Star_rating  66 non-null     int64  
 4   In_stock     66 non-null     bool   
 5   Category     66 non-null     str    
dtypes: bool(1), float64(2), int64(1), str(2)
memory usage: 2.8 KB


<StringArray>
[           'travel_2',  'science-fiction_16', 'sports-and-games_17',
          'science_22',       'psychology_26',         'business_35',
         'suspense_44']
Length: 7, dtype: str

In [45]:
for Category in Categories:
    cursor.execute("INSERT OR IGNORE INTO Categories(Name) VALUES(?)",(Category,))
conn.commit()

In [46]:
cursor.execute("PRAGMA table_info(Books)")
print(cursor.fetchall())

[(0, 'ID', 'INTEGER', 0, None, 1), (1, 'Title', 'TEXT', 1, None, 0), (2, 'CategoryID', 'INTEGER', 1, None, 0), (3, 'Price_GBP', 'REAL', 0, None, 0), (4, 'Price_INR', 'REAL', 0, None, 0), (5, 'Star_rating', 'INTEGER', 0, None, 0), (6, 'in_stock', 'INTEGER', 0, None, 0)]


In [47]:
cursor.execute('Select ID,Name from Categories')
category_map = {name: id for id, name in cursor.fetchall()}
books["CategoryID"] = books["Category"].map(category_map)

In [48]:
books_sql = books[[
        "Title",
        "CategoryID",
        "Price_GBP",
        "Price_INR",
        "Star_rating",
        "In_stock"]]
books_sql.to_sql(
    "Books",
    conn,
    if_exists="append",
    index=False
)
conn.commit()


In [49]:
cursor.execute("SELECT COUNT(*) FROM Books")
print(cursor.fetchone())

(198,)


In [50]:
queries = {}
outputs = {}

# Query 1
queries["Q1"] = """
SELECT * FROM Books;
"""
outputs["Q1"] = pd.read_sql(queries["Q1"], conn)

# Query 2
queries["Q2"] = """
SELECT * FROM Books
WHERE Star_rating >= 3;
"""
outputs["Q2"] = pd.read_sql(queries["Q2"], conn)

# Query 3
queries["Q3"] = """
SELECT DISTINCT Name
FROM Categories;
"""
outputs["Q3"] = pd.read_sql(queries["Q3"], conn)

# Query 4
queries["Q4"] = """
SELECT *
FROM Books
WHERE Price_GBP BETWEEN 20 AND 40;
"""
outputs["Q4"] = pd.read_sql(queries["Q4"], conn)

# Query 5
queries["Q5"] = """
SELECT *
FROM Books
WHERE Star_rating IN (4,5);
"""
outputs["Q5"] = pd.read_sql(queries["Q5"], conn)

# Query 6 (JOIN)
queries["Q6"] = """
SELECT
    b.Title,
    c.Name AS Category,
    b.Price_GBP,
    b.Price_INR,
    b.Star_rating,
    b.in_stock
FROM Books b
JOIN Categories c
ON b.CategoryID = c.ID
ORDER BY b.Title;
"""
outputs["Q6"] = pd.read_sql(queries["Q6"], conn)

# Query 7 : ORDER BY
queries["Q7"] = """
SELECT *
FROM Books
WHERE Star_rating >= 3
ORDER BY Star_rating DESC;
"""
outputs["Q7"] = pd.read_sql(queries["Q7"], conn)

# Query 8 : LIMIT + IN (subquery)
queries["Q8"] = """
SELECT *
FROM Books
WHERE ID IN (
    SELECT MIN(ID)
    FROM Books
    GROUP BY CategoryID, Star_rating
)
LIMIT 10;
"""
outputs["Q8"] = pd.read_sql(queries["Q8"], conn)

# Query 9 : IN (subquery)
queries["Q9"] = """
SELECT *
FROM Books
WHERE ID IN (
    SELECT MIN(ID)
    FROM Books
    GROUP BY CategoryID, Star_rating
);
"""
outputs["Q9"] = pd.read_sql(queries["Q9"], conn)

In [51]:
for key in queries:
    print("=" * 70)
    print(key)
    print("SQL Query:")
    print(queries[key])

    print("\nOutput:")
    print(outputs[key])

Q1
SQL Query:

SELECT * FROM Books;


Output:
      ID                                              Title  CategoryID  \
0      1                            It's Only the Himalayas           1   
1      2  Full Moon over Noah’s Ark: An Odyssey to Mount...           1   
2      3  See America: A Celebration of Our National Par...           1   
3      4  Vagabonding: An Uncommon Guide to the Art of L...           1   
4      5                               Under the Tuscan Sun           1   
..   ...                                                ...         ...   
193  194  The E-Myth Revisited: Why Most Small Businesse...           6   
194  195                                 Rich Dad, Poor Dad           6   
195  196  The Lean Startup: How Today's Entrepreneurs Us...           6   
196  197                                             Rework           6   
197  198               Silence in the Dark (Logan Point #4)           7   

     Price_GBP  Price_INR  Star_rating  in_stock  
0 

In [52]:
books_df = pd.read_sql("SELECT * FROM Books", conn)
categories_df = pd.read_sql("SELECT * FROM Categories", conn)

merge_df = pd.merge(
    books_df,
    categories_df,
    left_on="CategoryID",
    right_on="ID",
    how="inner"
)

merge_df = merge_df[[
    "Title",
    "Name",
    "Price_GBP",
    "Price_INR",
    "Star_rating",
    "in_stock"
]]

merge_df = merge_df.rename(columns={"Name": "Category"})
merge_df = merge_df.sort_values(
    ["Title", "Category"]
).reset_index(drop=True)

In [53]:
sql_join = outputs["Q6"].copy()

In [54]:
sql_join = sql_join.sort_values(
    ["Title", "Category"]
).reset_index(drop=True)
print("\nPandas Merge Result")
print(merge_df)
print("\nAre both outputs equivalent?")
print(sql_join.equals(merge_df))


Pandas Merge Result
                                                 Title            Category  \
0                   1,000 Places to See Before You Die            travel_2   
1                   1,000 Places to See Before You Die            travel_2   
2                   1,000 Places to See Before You Die            travel_2   
3             8 Keys to Mental Health Through Exercise       psychology_26   
4             8 Keys to Mental Health Through Exercise       psychology_26   
..                                                 ...                 ...   
193  Vagabonding: An Uncommon Guide to the Art of L...            travel_2   
194  Vagabonding: An Uncommon Guide to the Art of L...            travel_2   
195  William Shakespeare's Star Wars: Verily, A New...  science-fiction_16   
196  William Shakespeare's Star Wars: Verily, A New...  science-fiction_16   
197  William Shakespeare's Star Wars: Verily, A New...  science-fiction_16   

     Price_GBP  Price_INR  Star_rating  in

In [55]:
if not sql_join.equals(merge_df):
    print("\nDifferences:")
    print(sql_join.compare(merge_df))

In [56]:
print(sql_join.columns)
print(merge_df.columns)

Index(['Title', 'Category', 'Price_GBP', 'Price_INR', 'Star_rating',
       'in_stock'],
      dtype='str')
Index(['Title', 'Category', 'Price_GBP', 'Price_INR', 'Star_rating',
       'in_stock'],
      dtype='str')
